# Google Play: Game Audit (notebook 1b)
**Question 1:** are the apps in our 16 "game" categories actually games?
**Question 2:** can the real games inside **Sports** be separated from the non-game Sports apps?

The file has no field saying "this is a game". The only evidence is the **category the developer chose**. This notebook measures how reliable that is by drawing a random sample and judging each app by eye.

**Part A** draws the sample and writes `game_audit_sample.csv` (and the category sizes). **You (or Claude, with you checking) fill the `is_game` column.** **Part B** reads the filled file and reports the share of real games per category with a 95% interval, and tests a simple name rule for Sports.

Judging rule for `is_game`: **1** = a playable game; **0** = not a game (news, scores, tracker, forum, schedule, tips, tool); **?** = cannot tell from the name. Judge from the app name only. Anything doubtful goes to `?`, not 1.

In [1]:
import os, re, json
import numpy as np, pandas as pd
DATA_PATH = 'dataset_playstore1.csv'
MIN_RATINGS = 50          # same floor as notebook 1: we audit the population that would be labelled
PER_CATEGORY = 50         # sample size per game category
SPORTS_SAMPLE = 300
SEED = 7
GAME_CATS = {'action','adventure','arcade','board','card','casino','casual','educational','music','puzzle','racing','role playing','simulation','strategy','trivia','word'}
USE = ['App Name','App Id','Category','Rating','Rating Count']
parts=[]
for ch in pd.read_csv(DATA_PATH, usecols=USE, chunksize=300_000, low_memory=False):
    c = ch['Category'].astype(str).str.strip().str.lower()
    ch = ch[c.isin(GAME_CATS | {'sports'})].copy(); ch['cat'] = c[ch.index]
    ch['Rating'] = pd.to_numeric(ch['Rating'], errors='coerce'); ch['Rating Count'] = pd.to_numeric(ch['Rating Count'], errors='coerce')
    parts.append(ch[(ch['Rating']>0) & (ch['Rating Count']>=MIN_RATINGS)])
pop = pd.concat(parts, ignore_index=True)
print(f'apps that would be labelled (game categories + Sports): {len(pop):,}')
pop_sizes = pop['cat'].value_counts().rename_axis('category').reset_index(name='population')
pop_sizes.to_csv('game_audit_population.csv', index=False)
pop_sizes

apps that would be labelled (game categories + Sports): 121,705


,category,population
0,puzzle,15071
1,simulation,13387
2,casual,13169
3,sports,12302
4,arcade,10381
5,action,9867
6,adventure,9118
7,role playing,6126
8,educational,5606
9,racing,4530


In [2]:
# A simple name rule for Sports: flag words that usually mean "not a game". Tested in Part B, not trusted before.
NONGAME_WORDS = ['score','scores','news','live','fantasy','prediction','predictions','tips','schedule','tracker','forum','workout',
                 'fitness','training','coach','stream','radio','betting','odds','official','fixtures','results','stats','table','calendar','club','society']
rx = re.compile(r'\b(' + '|'.join(NONGAME_WORDS) + r')\b', re.I)
def rule_nongame(name): return bool(rx.search(str(name)))

rng = np.random.default_rng(SEED)
samples=[]
for cat, g in pop.groupby('cat'):
    n = SPORTS_SAMPLE if cat=='sports' else PER_CATEGORY
    samples.append(g.sample(min(n, len(g)), random_state=int(rng.integers(1e9))))
S = pd.concat(samples, ignore_index=True)
S['rule_says_nongame'] = S['App Name'].map(rule_nongame).astype(int)
out = S[['App Id','App Name','cat','Rating Count','rule_says_nongame']].rename(columns={'App Id':'app_id','App Name':'app_name','cat':'category','Rating Count':'rating_count'})
out = out.sample(frac=1, random_state=1).reset_index(drop=True)   # shuffled so categories are not judged in blocks
out['is_game'] = ''
out.to_csv('game_audit_sample.csv', index=False)
print(f'wrote game_audit_sample.csv with {len(out)} apps ({SPORTS_SAMPLE} Sports + up to {PER_CATEGORY} per game category)')
print('Next: fill the is_game column (1 / 0 / ?), save as game_audit_filled.csv, then run Part B.')
out.head(10)

wrote game_audit_sample.csv with 1100 apps (300 Sports + up to 50 per game category)
Next: fill the is_game column (1 / 0 / ?), save as game_audit_filled.csv, then run Part B.


,app_id,app_name,category,rating_count,rule_says_nongame,is_game
0,com.ucdistanbul.gtrdriftsimulator,GTR Drift Simulator,racing,472.0,0,
1,com.tpwar,Телепорты - Онлайн PvP,strategy,97.0,0,
2,boriol.learn.clocklite,Telling Time Kids 1st Grade,educational,674.0,0,
3,com.zn.monster,Legend of Darkness,role playing,22683.0,0,
4,com.atleticogoianiense,Atlético Clube Goianiense,sports,63.0,0,
5,air.apdev.Zilch,Zilch,board,215.0,0,
6,com.rusdev.bettingtips.footballprediction,Football Betting Tips : Free Football Prediction,sports,194.0,1,
7,bettips.multibettingtips.bestprediction.tennis...,Daily Betting Predictions Tips,sports,159.0,1,
8,it.fantaking.kickest,Kickest - Advanced Fantasy Football,sports,793.0,1,
9,com.bubble.pop.wintrino,Bubble Pop,board,3752.0,0,


## Part B: results (run after `game_audit_filled.csv` exists)

In [3]:
FILLED = 'game_audit_filled.csv'
if not os.path.exists(FILLED):
    print('game_audit_filled.csv not found yet. Fill is_game in game_audit_sample.csv and save it under that name.')
else:
    F = pd.read_csv(FILLED, dtype={'is_game': str}); F['is_game'] = F['is_game'].astype(str).str.strip()
    unknown = (F['is_game'] == '?').sum(); blank = (~F['is_game'].isin(['0','1','?'])).sum()
    print(f'judged 1/0: {F["is_game"].isin(["0","1"]).sum()} | unknown (?): {unknown} | blank or invalid: {blank}')
    J = F[F['is_game'].isin(['0','1'])].copy(); J['g'] = J['is_game'].astype(int)
    def wilson(k, n, z=1.96):
        if n == 0: return (np.nan, np.nan)
        p = k/n; d = 1 + z*z/n; c = p + z*z/(2*n); a = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
        return ((c-a)/d, (c+a)/d)
    pop_sizes = pd.read_csv('game_audit_population.csv').set_index('category')['population']
    rows=[]
    for cat, g in J.groupby('category'):
        k, n = int(g['g'].sum()), len(g); lo, hi = wilson(k, n)
        rows.append({'category': cat, 'judged': n, 'games': k, 'share_game': round(k/n, 3), 'ci_low': round(lo, 3), 'ci_high': round(hi, 3), 'population': int(pop_sizes.get(cat, 0))})
    R = pd.DataFrame(rows).sort_values('share_game'); display(R)
    G = R[R['category'] != 'sports']
    est = (G['share_game'] * G['population']).sum() / G['population'].sum()
    print(f'Estimated share of real games across the 16 game categories (weighted by size): {est*100:.1f}%')

game_audit_filled.csv not found yet. Fill is_game in game_audit_sample.csv and save it under that name.


In [4]:
if os.path.exists(FILLED):
    Sp = J[J['category'] == 'sports']
    print(f'Sports judged: {len(Sp)} | real games: {Sp["g"].sum()} ({Sp["g"].mean()*100:.1f}%)')
    tab = pd.crosstab(Sp['rule_says_nongame'], Sp['g'], rownames=['rule says non-game'], colnames=['really a game'])
    display(tab)
    kept = Sp[Sp['rule_says_nongame'] == 0]
    if len(kept):
        prec = kept['g'].mean(); rec = kept['g'].sum() / max(Sp['g'].sum(), 1)
        print(f'If we KEEP Sports apps the rule does not flag: precision (kept that are games) {prec*100:.1f}% | recall (games that survive) {rec*100:.1f}%')
    print('Compare precision with the game categories above. Only include rule-filtered Sports if its precision is close to theirs.')
    print('\nRule mistakes to read (flagged but a game, and kept but not a game):')
    display(Sp[(Sp['rule_says_nongame'] == 1) & (Sp['g'] == 1)][['app_name']].head(15))
    display(Sp[(Sp['rule_says_nongame'] == 0) & (Sp['g'] == 0)][['app_name']].head(15))
    print('\nNon-game apps found in the 16 game categories (read these to see what leaks in):')
    display(J[(J['category'] != 'sports') & (J['g'] == 0)][['category', 'app_name']].head(40))